In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
import os

os.getenv("TAVILY_API_KEY")

'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'

In [ ]:
from langchain_tavily import TavilySearch
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

In [7]:
llm = ChatOllama(model="minimax-m3:cloud")

Create Tool


In [8]:
tool = TavilySearch(
    max_results=1,
    topic="general",
)

In [9]:
llm_with_tools = llm.bind_tools([tool])

In [ ]:
llm_with_tools.invoke({"query": "What is today's weather in bhopal?"})

{'query': "What is today's weather in bhopal?",
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.climate-data.org/asia/india/madhya-pradesh/bhopal-2833/t/july-7',
   'title': 'Weather Bhopal in July 2026: Temperature & Climate',
   'content': '| 30. July | Heavy rain | 30 °C 86 °F | 25 °C 77 °F | 95 % | 8 km/h 5 mph | 103mm 4.06 in | 89% |\n| 31. July | Heavy rain | 29 °C 84.2 °F | 24 °C 75.2 °F | 90 % | 17 km/h 11 mph | 29mm 1.14 in | 87% |\n| 1. August | Thunderstorm with rain | 29 °C 84.2 °F | 23 °C 73.4 °F | 80 % | 17 km/h 11 mph | 14mm 0.55 in | 88% |',
   'score': 0.9338243,
   'raw_content': None}],
 'response_time': 1.97,
 'request_id': 'd5d012ab-057b-444c-817a-680a8f6e11ed'}

Create Agent


In [ ]:
agent = create_agent(
    model=llm,
    tools=[tool],
    system_prompt="You are a agent and can search any question.",
)

In [11]:
question = "What is the AQI level of bhopal?"

In [14]:
res = agent.invoke({"messages": [{"role": "user", "content": question}]})
res

{'messages': [HumanMessage(content='What is the AQI level of bhopal?', additional_kwargs={}, response_metadata={}, id='8e7a9151-9ec2-4544-86f5-260e8c4fc3e2'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'minimax-m3', 'created_at': '2026-07-30T16:57:12.703160825Z', 'done': True, 'done_reason': 'tool_calls', 'total_duration': 1157564109, 'load_duration': None, 'prompt_eval_count': None, 'prompt_eval_duration': None, 'eval_count': 94, 'eval_duration': None, 'logprobs': None, 'model_name': 'minimax-m3', 'model_provider': 'ollama'}, id='lc_run--019fb3f5-605e-7b51-b285-1817a8e9a72e-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Bhopal AQI level today air quality index', 'search_depth': 'basic', 'topic': 'general'}, 'id': 'ba806173-e38e-453f-97ea-0f1c26f6bb67', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='{"query": "Bhopal AQI level today air quality index", "follow_up_questions": null, "answer": null, "images": [], "results

In [16]:
print(res["messages"][-1].content)

The **current AQI of Bhopal is 75**, which falls in the **"Moderate"** category (51–100).

Here are some additional details:
- **PM2.5 level:** 23 µg/m³
- **PM10 level:** 27 µg/m³
- **AQI category:** Moderate
- **Health advisory:** Air quality is acceptable for most people, but sensitive individuals (those with asthma or allergies) may experience mild discomfort and should consider reducing prolonged outdoor exertion, especially during peak pollution hours.

Note: AQI values change frequently throughout the day based on real-time air quality readings. For live updates, you can check sources like [India Today's Bhopal AQI page](https://www.indiatoday.in/weather/bhopal-air-quality-index-aqi-today).


Create Memory agent


In [18]:
from langgraph.checkpoint.memory import InMemorySaver

In [19]:
tool_memory = TavilySearch(
    max_results=1,
    topic="general",
)

In [20]:
agent = create_agent(
    model=llm,
    tools=[tool],
    system_prompt="You are a agent and can search any question.",
    checkpointer=InMemorySaver(),
)

In [ ]:
# question_memory= "Who is PM of INDIA?"
question_memory = "What is his age?"

In [ ]:
res = agent.invoke(
    {"messages": [{"role": "user", "content": question_memory}]},
    {"configurable": {"thread_id": "test_memory"}},
)

res["messages"][-1].content

'**Narendra Modi** is **75 years old**. He was born on **17 September 1950** in Vadnagar, Bombay State (now Gujarat), India.'